# Smart Building Energy — K-Means vs Decision Tree vs KNN
**Dataset:** UMass Smart* Home A — Real electrical circuit data (2012)  
**Source:** https://traces.cs.umass.edu/docs/traces/smartstar/  
**Environment:** Google Colab — CSV stored in Google Drive (`DS540_ML_Project/`)  
**Algorithms:** K-Means (unsupervised) · Decision Tree (supervised + Grid Search) · KNN (supervised + Cross-Validation)  

| Property | Value |
|---|---|
| Records | 132,480 (1-minute intervals) |
| Features | 25 real appliance circuits + 4 temporal features |
| Date range | Apr 30 – Jul 31, 2012 |
| Target | energy_tier: Low / Medium / High |
| Tuning | Elbow Method (K-Means) · Grid Search CV (Decision Tree) · 5-Fold CV (KNN) |
| CSV location | `/content/drive/MyDrive/DS540_ML_Project/smartstar_homeA_processed.csv` |

> ⚠️ **Before running:** Upload `smartstar_homeA_processed.csv` to your Google Drive inside a folder named `DS540_ML_Project`. Then run Section 2 first — it will prompt you to authorise Drive access.

---
## Notebook Structure
1. Import Libraries  
2. Mount Google Drive & Load Dataset  
3. Exploratory Data Analysis (EDA)  
4. Preprocessing  
5. K-Means Clustering  
6. Decision Tree + Grid Search  
7. K-Nearest Neighbours + Cross-Validation  
8. Final Comparison  
9. Conclusions  
---

## 1. Import Libraries
> **Why:** We collect every library we need in one place at the top so the notebook runs cleanly from top to bottom without import errors mid-way. `GridSearchCV` is new here — it automates the search over every combination of Decision Tree hyperparameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# ── Algorithms ────────────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier

# ── Model Selection & Tuning ──────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV          # <-- NEW: exhaustive hyperparameter search
)

# ── Evaluation Metrics ────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    silhouette_score, ConfusionMatrixDisplay
)

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.family'       : 'DejaVu Sans',
    'axes.titlesize'    : 12,
    'axes.labelsize'    : 11
})

LABELS  = ['Low', 'Medium', 'High']
PALETTE = {'Low': '#1D9E75', 'Medium': '#BA7517', 'High': '#993C1D'}
CLR3    = ['#1D9E75', '#BA7517', '#993C1D']

print('All libraries imported successfully.')

---
## 2. Mount Google Drive & Load Dataset
> **Why:** The dataset CSV is stored in your Google Drive folder `DS540_ML_Project`. In Colab, we must mount Drive first using `drive.mount()` before we can access any files in it. This cell handles both mounting and loading in one place.

In [ ]:
# ── Step 1: Mount Google Drive (run this cell first in Colab) ───────────────
# WHY: The CSV file lives in your Google Drive folder DS540_ML_Project.
# We must mount Drive first so Colab can read files from it.
# After running, click the link, sign in, and paste the authorisation code.
from google.colab import drive
drive.mount('/content/drive')

# ── Step 2: Set the CSV path to your Drive location ──────────────────────────
# WHY: This matches the exact folder structure visible in your Colab file panel:
#   MyDrive → DS540_ML_Project → smartstar_homeA_processed.csv
# If you rename the folder, update CSV_PATH accordingly.
CSV_PATH = '/content/drive/MyDrive/DS540_ML_Project/smartstar_homeA_processed.csv'

# ── Step 3: Load the dataset ─────────────────────────────────────────────────
# The CSV was built from 91 raw daily Smart* files, pivoted so each circuit
# becomes a column, resampled to 1-minute intervals, and forward-filled.
df = pd.read_csv(CSV_PATH, parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f'Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range     : {df["datetime"].min()} → {df["datetime"].max()}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'\nColumn list:')
print(list(df.columns))
df.head(3)

---
## 3. Exploratory Data Analysis (EDA)
> **Why:** EDA is the essential first step before any modelling. We need to understand the shape of the data, identify patterns, spot skewness, and see which features carry the most signal. Every preprocessing decision in Section 4 is justified by what we find here.

In [ ]:
# ── 3.1 Define feature groups ─────────────────────────────────────────────────
# WHY: Separating circuit columns from temporal columns makes it easier to
# apply different transformations to each group and track what we're using.

CIRCUIT_COLS = [
    'BedroomLights','BedroomOutlets','CellarLights','CellarOutlets',
    'CounterOutlets1','CounterOutlets2','DiningRoomOutlets','DisposalDishwasher',
    'Dryer','DuctHeaterHRV','FridgeRange','FurnaceHRV',
    'GuestBathHallLights','GuestBathOutlets','KitchenLights','KitchenOutlets',
    'LivingRoomOutlets','LivingRoomPatioLights','MasterBathOutlets',
    'MasterLights','MasterOutlets','Microwave','OfficeOutlets',
    'OutsideOutlets','WashingMachine'
]
TEMPORAL_COLS = ['hour', 'dayofweek', 'month', 'is_weekend']

print('=== Dataset Overview ===')
print(f'Total records         : {len(df):,}')
print(f'Circuit features      : {len(CIRCUIT_COLS)}')
print(f'Temporal features     : {len(TEMPORAL_COLS)}')
print(f'Target classes        : {LABELS}')
print(f'Missing values        : {df[CIRCUIT_COLS].isnull().sum().sum()}')
print(f'\nDescriptive stats — total_load_watts:')
print(df['total_load_watts'].describe().round(2))
print(f'\nTarget class distribution:')
print(df['energy_tier'].value_counts())

In [ ]:
# ── 3.2 Target distribution: raw + log + class balance ───────────────────────
# WHY: The raw histogram reveals right-skew (many low-watt minutes, few spikes).
# The log-transformed view shows the underlying shape more clearly.
# The bar chart confirms near-balanced classes — no resampling needed.

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['total_load_watts'], bins=80, color='#185FA5',
             edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Total Home Load (Watts)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Raw Distribution — Right-skewed')

axes[1].hist(np.log1p(df['total_load_watts']), bins=80, color='#534AB7',
             edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('log(1 + Watts)')
axes[1].set_title('Log-Transformed — More symmetric')

tier_counts = df['energy_tier'].value_counts().reindex(LABELS)
bars = axes[2].bar(tier_counts.index, tier_counts.values,
                   color=[PALETTE[t] for t in tier_counts.index],
                   edgecolor='white', linewidth=0.5)
axes[2].bar_label(bars, fmt='%d', padding=3, fontsize=9)
axes[2].set_ylabel('Count')
axes[2].set_title('Class Balance — Nearly equal tiers')

plt.suptitle('EDA Step 1 — Target Variable Overview', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Temporal patterns: hour of day + day of week ─────────────────────────
# WHY: Energy consumption follows human activity patterns. If we can see clear
# differences by hour/day, it confirms that temporal features will be strong
# predictors for both Decision Tree and KNN — worth including in the model.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly = df.groupby('hour')['total_load_watts'].mean()
axes[0].bar(hourly.index, hourly.values, color='#185FA5',
            edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Load (Watts)')
axes[0].set_title('Hourly Pattern — Peak usage 17:00-21:00')
axes[0].set_xticks(range(0, 24, 2))

day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily = df.groupby('dayofweek')['total_load_watts'].mean()
bar_colors = ['#BA7517' if i >= 5 else '#185FA5' for i in daily.index]
axes[1].bar(daily.index, daily.values, color=bar_colors,
            edgecolor='white', linewidth=0.4)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)
axes[1].set_ylabel('Avg Load (Watts)')
axes[1].set_title('Daily Pattern — Higher on weekends (orange)')

plt.suptitle('EDA Step 2 — Temporal Patterns (justify adding hour + dayofweek features)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Per-circuit average consumption ──────────────────────────────────────
# WHY: Identifies which appliances dominate energy consumption.
# Circuits with near-zero average (e.g. outside lights) add noise but
# little signal — this justifies dropping low-variance features in preprocessing.

circuit_means = df[CIRCUIT_COLS].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(circuit_means.index, circuit_means.values,
               color='#1D9E75', edgecolor='white', linewidth=0.4)
ax.bar_label(bars, fmt='%.1f W', padding=3, fontsize=8)
ax.set_xlabel('Average Power (Watts)')
ax.set_title('EDA Step 3 — Average Load per Circuit\n(High-bar circuits will dominate feature importance)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Boxplot: top circuits by energy tier ─────────────────────────────────
# WHY: If a circuit shows clearly different distributions across the three
# energy tiers, it is a strong discriminating feature for classification.
# This gives us intuition about which circuits the Decision Tree will split on.

top_circuits = df[CIRCUIT_COLS].mean().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i, col in enumerate(top_circuits):
    data_by_tier = [df[df['energy_tier'] == t][col].values for t in LABELS]
    bp = axes[i].boxplot(data_by_tier, patch_artist=True, labels=LABELS,
                          medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], CLR3):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Watts')

plt.suptitle('EDA Step 4 — Top Circuits vs Energy Tier\n(Separation between boxes = strong classification signal)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Correlation heatmap ───────────────────────────────────────────────────
# WHY: Highly correlated features carry redundant information.
# If two circuits have correlation > 0.9, we may consider keeping only one.
# Also shows which circuits are most correlated with the total load.

top12 = df[CIRCUIT_COLS].mean().nlargest(12).index.tolist()
corr  = df[top12 + ['total_load_watts']].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title('EDA Step 5 — Correlation Matrix: Top 12 Circuits + Total Load\n(Values near ±1 = highly correlated pairs)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.7 Variance analysis — detect low-information circuits ──────────────────
# WHY: Features with near-zero variance contribute no information to any
# algorithm. K-Means is especially sensitive to this because low-variance
# features create 'dead dimensions' that distort distance calculations.
# We will remove any feature with std < 1 Watt in Section 4.

circuit_std = df[CIRCUIT_COLS].std().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors_var = ['#E24B4A' if v < 5 else '#185FA5' for v in circuit_std.values]
bars = ax.barh(circuit_std.index, circuit_std.values,
               color=colors_var, edgecolor='white', linewidth=0.4)
ax.axvline(x=5, color='#E24B4A', linestyle='--', linewidth=1.2, label='Drop threshold (std < 5W)')
ax.set_xlabel('Standard Deviation (Watts)')
ax.set_title('EDA Step 6 — Variance per Circuit\n(Red bars = near-constant, low-signal features)')
ax.legend()
plt.tight_layout()
plt.show()

low_var = circuit_std[circuit_std < 5].index.tolist()
print(f'Low-variance circuits to consider dropping (std < 5W): {low_var}')

In [ ]:
# ── 3.8 Outlier detection — IQR method ───────────────────────────────────────
# WHY: Extreme outliers can skew K-Means centroids and distort KNN distances.
# We use the IQR method (1.5× rule) to count outliers per circuit.
# We won't delete them — just understand their scale before capping in Section 4.

outlier_counts = {}
for col in CIRCUIT_COLS:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_counts[col] = n_out

outlier_series = pd.Series(outlier_counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
outlier_series.sort_values().plot(kind='barh', ax=ax,
                                   color='#D85A30', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Number of Outlier Readings (IQR method)')
ax.set_title('EDA Step 7 — Outliers per Circuit\n(Will be capped at 99th percentile in preprocessing)')
plt.tight_layout()
plt.show()

print('Top 5 circuits with most outliers:')
print(outlier_series.head())

---
## 4. Preprocessing
> **Why this order matters:** Outlier capping → low-variance feature removal → scaling → PCA → train/test split. Each step builds on the previous. Scaling must happen AFTER removing features and BEFORE PCA/modelling. The train/test split is always last to prevent data leakage.

In [ ]:
# ── 4.1 Outlier capping at 99th percentile (Winsorization) ───────────────────
# WHY: Rather than deleting outlier rows (which loses 1-minute snapshots of
# real events like a dryer running), we cap extreme values at the 99th
# percentile. This keeps all 132,480 records while reducing the influence
# of extreme spikes on K-Means centroids and KNN distances.

df_clean = df.copy()

for col in CIRCUIT_COLS:
    cap = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(upper=cap)

print('Outlier capping complete (99th percentile Winsorization).')
print(f'Records retained : {len(df_clean):,} (no rows deleted)')

# Verify — compare max before/after on the most extreme circuit
most_extreme = outlier_series.index[0]
print(f'\nExample — {most_extreme}:')
print(f'  Before cap: max = {df[most_extreme].max():.1f} W')
print(f'  After  cap: max = {df_clean[most_extreme].max():.1f} W')

In [ ]:
# ── 4.2 Remove low-variance features ─────────────────────────────────────────
# WHY: Features with very low variance are near-constant across all records.
# They contribute no discriminating power to any classifier, but they DO add
# noise to K-Means distance calculations and slow down KNN lookups.
# Threshold: std < 5W means the circuit rarely changes — drop it.

circuit_std_clean = df_clean[CIRCUIT_COLS].std()
LOW_VAR_THRESHOLD = 5  # Watts

DROP_CIRCUITS = circuit_std_clean[circuit_std_clean < LOW_VAR_THRESHOLD].index.tolist()
KEEP_CIRCUITS = [c for c in CIRCUIT_COLS if c not in DROP_CIRCUITS]

print(f'Low-variance circuits dropped (std < {LOW_VAR_THRESHOLD}W): {DROP_CIRCUITS}')
print(f'Circuits retained: {len(KEEP_CIRCUITS)}')
print(f'Retained: {KEEP_CIRCUITS}')

In [ ]:
# ── 4.3 Define feature matrix and target ──────────────────────────────────────
# WHY: We combine circuit features (what appliances are running) with
# temporal features (what time it is) because both carry complementary
# information. The EDA showed strong hourly and daily patterns.
# We exclude 'Grid' (that IS the total load — using it would be data leakage).

FEATURE_COLS = KEEP_CIRCUITS + TEMPORAL_COLS
TARGET_COL   = 'energy_tier'

X = df_clean[FEATURE_COLS].copy()
y = df_clean[TARGET_COL].copy()

print(f'Feature matrix shape : {X.shape}')
print(f'Target distribution  :')
print(y.value_counts())
print(f'\nFeature columns used ({len(FEATURE_COLS)}):')
print(FEATURE_COLS)

In [ ]:
# ── 4.4 StandardScaler ────────────────────────────────────────────────────────
# WHY: Circuit values range from 0-3000 Watts, while temporal features like
# 'hour' range 0-23 and 'is_weekend' is 0/1. Without scaling, high-watt
# circuits completely dominate K-Means distance and KNN distance calculations.
# StandardScaler transforms each feature to mean=0, std=1, giving every
# feature equal influence. Decision Tree does NOT need scaling (it uses
# threshold comparisons) but we scale anyway for consistency.

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

print('StandardScaler applied.')
print(f'Mean across all features : {X_scaled.mean().mean():.6f}  (should be ≈ 0)')
print(f'Std  across all features : {X_scaled.std().mean():.6f}  (should be ≈ 1)')
print(f'\nBefore scaling — FurnaceHRV range: {X["FurnaceHRV"].min():.1f} to {X["FurnaceHRV"].max():.1f} W')
print(f'After  scaling — FurnaceHRV range: {X_scaled["FurnaceHRV"].min():.2f} to {X_scaled["FurnaceHRV"].max():.2f}')

In [ ]:
# ── 4.5 Before vs After Scaling Visualisation ─────────────────────────────────
# WHY: Visually confirms that scaling has equalised the feature ranges.
# The 'before' boxplot shows wild differences in scale across circuits.
# The 'after' boxplot shows all features now centered near zero.

top6 = df_clean[KEEP_CIRCUITS].mean().nlargest(6).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X[top6].boxplot(ax=axes[0], patch_artist=True,
                boxprops=dict(facecolor='#E6F1FB', color='#185FA5'),
                medianprops=dict(color='#185FA5'))
axes[0].set_title('Before Scaling — Raw Watts\n(Huge scale differences)')
axes[0].set_ylabel('Watts')
axes[0].tick_params(axis='x', rotation=45)

X_scaled[top6].boxplot(ax=axes[1], patch_artist=True,
                        boxprops=dict(facecolor='#E1F5EE', color='#1D9E75'),
                        medianprops=dict(color='#1D9E75'))
axes[1].set_title('After StandardScaler — Standardised\n(All features on same scale)')
axes[1].set_ylabel('Standard deviations from mean')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Preprocessing Step 1 — Effect of StandardScaler on Top 6 Circuits', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.6 PCA for visualisation (2 components) ─────────────────────────────────
# WHY: PCA compresses our 29-dimensional feature space into 2 dimensions so
# we can plot and visually inspect cluster separation. We are NOT using PCA
# as a preprocessing step for the algorithms — only for the 2D scatter plots.
# This is important: we fit PCA on the FULL scaled dataset before splitting.

pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

ev = pca.explained_variance_ratio_
print(f'PCA explained variance:')
print(f'  PC1: {ev[0]*100:.1f}%')
print(f'  PC2: {ev[1]*100:.1f}%')
print(f'  Total: {sum(ev)*100:.1f}%  of information preserved in 2D')

In [ ]:
# ── 4.7 Stratified Train/Test Split (80/20) ───────────────────────────────────
# WHY: We hold out 20% of data as a test set that the models NEVER see during
# training or hyperparameter tuning. 'stratify=y' ensures each split has the
# same proportion of Low/Medium/High — important for fair evaluation.
# This split is done LAST to prevent data leakage from scaler or PCA.

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train set : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Test  set : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'\nTrain class distribution (stratified):')
print(y_train.value_counts())
print(f'\nTest class distribution (stratified):')
print(y_test.value_counts())

---
## 5. Algorithm 1 — K-Means Clustering
> **Why K-Means:** It is an unsupervised algorithm — it finds natural groupings in the data WITHOUT being told the energy tiers. This makes it valuable as a baseline: if K-Means discovers clusters that align with our manually defined Low/Medium/High tiers, it validates that the tiers represent real structure in the data, not arbitrary labels.

> **Tuning method:** Elbow Method + Silhouette Score. These are the standard tools for selecting K in an unsupervised setting where there is no label to optimise against.

In [ ]:
# ── 5.1 Elbow Method — find optimal K ────────────────────────────────────────
# WHY: The 'elbow' is the point where adding more clusters stops reducing
# inertia significantly. Before the elbow: merging clusters, too coarse.
# After the elbow: splitting real clusters, overfitting the structure.
# We test K=2..10 and look for the bend.

inertia_list    = []
silhouette_list = []
K_RANGE = range(2, 11)

print('Computing K-Means for K = 2 to 10 ...')
for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertia_list.append(km.inertia_)
    sil = silhouette_score(X_scaled, lbl, sample_size=5000, random_state=42)
    silhouette_list.append(sil)
    print(f'  K={k:2d} | Inertia={km.inertia_:>14,.0f} | Silhouette={sil:.4f}')

best_sil_k = K_RANGE[np.argmax(silhouette_list)]
print(f'\nBest K by Silhouette = {best_sil_k}')

In [ ]:
# ── 5.2 Plot Elbow + Silhouette ────────────────────────────────────────────────
# WHY: We plot both metrics together to make the K=3 decision defensible.
# Inertia alone can suggest K=2 (always a valid partition). Silhouette
# penalises poor cluster separation and gives a more balanced view.
# K=3 matches our 3 energy tiers AND sits at the elbow bend.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(K_RANGE, inertia_list, marker='o', color='#185FA5', linewidth=2)
axes[0].axvline(x=3, linestyle='--', color='#993C1D', alpha=0.7, label='K=3 selected')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — bend at K=3')
axes[0].set_xticks(K_RANGE)
axes[0].legend()

axes[1].plot(K_RANGE, silhouette_list, marker='s', color='#1D9E75', linewidth=2)
axes[1].axvline(x=3, linestyle='--', color='#993C1D', alpha=0.7, label='K=3 selected')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score  (higher = better)')
axes[1].set_title('Silhouette Score — peak near K=3')
axes[1].set_xticks(K_RANGE)
axes[1].legend()

plt.suptitle('K-Means Tuning — Elbow Method + Silhouette Score', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Train final K-Means with K=3 ─────────────────────────────────────────
# WHY K=3: Both the Elbow and Silhouette agree on K=3. Additionally, K=3
# aligns with our domain knowledge — we have three defined energy tiers.
# n_init=10 restarts the algorithm 10 times with different centroid seeds
# and keeps the best run — this avoids the local-minimum problem in K-Means.

OPTIMAL_K = 3

kmeans    = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)
df_clean  = df_clean.copy()
df_clean['km_cluster'] = km_labels

sil_final = silhouette_score(X_scaled, km_labels, sample_size=8000, random_state=42)

print(f'K-Means (K={OPTIMAL_K}, n_init=10)')
print(f'  Inertia          : {kmeans.inertia_:,.2f}')
print(f'  Silhouette Score : {sil_final:.4f}  (range −1 to +1, higher = better)')
for c, cnt in zip(*np.unique(km_labels, return_counts=True)):
    print(f'  Cluster {c}: {cnt:,} samples ({cnt/len(df_clean)*100:.1f}%)')

In [ ]:
# ── 5.4 Name clusters by mean energy ─────────────────────────────────────────
# WHY: K-Means assigns arbitrary cluster numbers (0,1,2). We map them to
# meaningful names by sorting clusters by their mean total load —
# lowest mean → Low-Usage, middle → Medium-Usage, highest → High-Usage.

cluster_profile = df_clean.groupby('km_cluster')['total_load_watts'].agg(['mean','median','std','count'])
print('Cluster Energy Profile (Watts):')
print(cluster_profile.round(1))

order = cluster_profile['mean'].sort_values().index.tolist()
cluster_name_map = {
    order[0]: 'Low-Usage',
    order[1]: 'Medium-Usage',
    order[2]: 'High-Usage'
}
df_clean['km_cluster_name'] = df_clean['km_cluster'].map(cluster_name_map)
print('\nCluster → name mapping:', cluster_name_map)

In [ ]:
# ── 5.5 PCA 2D cluster visualisation ─────────────────────────────────────────
# WHY: We project the 29-dimensional clusters down to 2D using PCA so we can
# visually inspect how well-separated the clusters are. We plot K-Means
# clusters side-by-side with the true energy tiers to see how well the
# unsupervised algorithm has rediscovered the labeled structure.

centroids_pca = pca.transform(kmeans.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c, color in enumerate(CLR3):
    mask = km_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, s=0.8, alpha=0.25, label=cluster_name_map[c])
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', s=150, marker='X', zorder=5, label='Centroids')
axes[0].set_title('K-Means Clusters (PCA 2D)\nColors = discovered clusters')
axes[0].set_xlabel(f'PC1 ({ev[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({ev[1]*100:.1f}% variance)')
axes[0].legend(markerscale=6, fontsize=8)

for tier, color in PALETTE.items():
    mask = (df_clean['energy_tier'] == tier).values
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, s=0.8, alpha=0.25, label=tier)
axes[1].set_title('True Energy Tiers (PCA 2D)\nColors = actual labels')
axes[1].set_xlabel(f'PC1 ({ev[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({ev[1]*100:.1f}% variance)')
axes[1].legend(markerscale=6, fontsize=8)

plt.suptitle('K-Means vs True Tiers — How well did unsupervised learning rediscover the labels?', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6 Cluster hourly heatmap ────────────────────────────────────────────────
# WHY: This reveals WHEN each cluster occurs during the day.
# If High-Usage cluster concentrates in evening hours and Low-Usage at night,
# it confirms that the clusters have a temporal interpretation —
# not just random groupings, but real behavioural patterns.

heatmap_data = df_clean.groupby(['km_cluster_name', 'hour'])['total_load_watts'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax,
            linewidths=0.3, annot=False,
            cbar_kws={'label': 'Avg Watts'})
ax.set_xlabel('Hour of Day')
ax.set_ylabel('K-Means Cluster')
ax.set_title('Hourly Load Heatmap per Cluster — Reveals time-of-day behaviour patterns')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.7 Cluster purity vs true labels ─────────────────────────────────────────
# WHY: Cluster purity bridges unsupervised and supervised evaluation.
# It answers: of the true labels inside each K-Means cluster,
# what fraction belong to the dominant class?
# High purity = K-Means clusters align well with the true energy tiers.

print('K-Means Cluster vs True Tier Cross-tabulation:')
crosstab = pd.crosstab(df_clean['km_cluster_name'], df_clean['energy_tier'], margins=True)
print(crosstab)
print()

total_correct = 0
for cluster in ['Low-Usage', 'Medium-Usage', 'High-Usage']:
    subset   = df_clean[df_clean['km_cluster_name'] == cluster]['energy_tier']
    majority = subset.value_counts().iloc[0]
    dominant = subset.value_counts().index[0]
    purity_c = majority / len(subset)
    total_correct += majority
    print(f'  {cluster:15s} → dominant tier: {dominant:8s} | purity = {purity_c:.4f}')

overall_purity = total_correct / len(df_clean)
print(f'\n  Overall Cluster Purity = {overall_purity:.4f}')
print(f'  Silhouette Score       = {sil_final:.4f}')

---
## 6. Algorithm 2 — Decision Tree + Grid Search
> **Why Decision Tree:** A Decision Tree builds explicit if/else rules from the features. It is the most interpretable supervised classifier — you can read the tree and understand exactly why a particular 1-minute window is classified as High energy. This is directly actionable for smart building automation.

> **Why Grid Search:** The Decision Tree has 4 interacting hyperparameters (max_depth, min_samples_split, min_samples_leaf, class_weight). A manual depth-only loop misses the interactions between these parameters. GridSearchCV tests every combination exhaustively and returns the globally optimal set.

In [ ]:
# ── 6.1 Define the hyperparameter grid ───────────────────────────────────────
# WHY each parameter:
#   max_depth         — controls tree complexity. Too deep = memorises training data.
#   min_samples_split — minimum records needed to split a node. Higher = simpler tree.
#   min_samples_leaf  — minimum records in a leaf. Prevents tiny, noisy leaves.
#   class_weight      — 'balanced' adjusts for any minor class imbalance automatically.
# GridSearchCV tests every combination (4×3×3×2 = 72 combinations × 5 folds = 360 fits).

param_grid_dt = {
    'max_depth'         : [4, 6, 8, 12],
    'min_samples_split' : [10, 20, 50],
    'min_samples_leaf'  : [5, 10, 20],
    'class_weight'      : ['balanced', None]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_dt = GridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=42),
    param_grid = param_grid_dt,
    cv         = cv_strategy,
    scoring    = 'f1_weighted',   # WHY f1_weighted: balances precision and recall across all 3 tiers
    n_jobs     = -1,              # WHY -1: uses all CPU cores for speed
    verbose    = 1,
    refit      = True             # WHY refit=True: auto-retrain best model on full train set
)

print(f'Grid size: {len(param_grid_dt["max_depth"])} depths × '
      f'{len(param_grid_dt["min_samples_split"])} splits × '
      f'{len(param_grid_dt["min_samples_leaf"])} leaves × '
      f'{len(param_grid_dt["class_weight"])} weights '
      f'= {4*3*3*2} combinations × 5 folds = {4*3*3*2*5} model fits')
print('\nRunning Grid Search ...')
grid_dt.fit(X_train, y_train)

print(f'\nBest parameters found:')
for k, v in grid_dt.best_params_.items():
    print(f'  {k:25s}: {v}')
print(f'\nBest CV F1-Score (weighted): {grid_dt.best_score_:.4f}')

In [ ]:
# ── 6.2 Visualise Grid Search results ────────────────────────────────────────
# WHY: Plotting the CV results lets us see how sensitive performance is to
# each hyperparameter. A flat curve means the parameter doesn't matter much.
# A sharp peak means the parameter is critical to tune correctly.

cv_results = pd.DataFrame(grid_dt.cv_results_)

# Plot mean CV F1 vs max_depth for each class_weight
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cw, color, label in [('balanced', '#1D9E75', 'class_weight=balanced'),
                           (None,       '#993C1D', 'class_weight=None')]:
    subset = cv_results[cv_results['param_class_weight'] == cw]
    depth_f1 = subset.groupby('param_max_depth')['mean_test_score'].max()
    axes[0].plot(depth_f1.index, depth_f1.values, marker='o',
                 color=color, label=label, linewidth=2)

axes[0].axvline(x=grid_dt.best_params_['max_depth'],
                linestyle='--', color='gray', alpha=0.6, label='Best depth')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Best CV F1-Score (weighted)')
axes[0].set_title('Grid Search — Depth vs F1 by class_weight')
axes[0].legend(fontsize=8)

# All 72 combinations sorted by CV score
top20 = cv_results.nlargest(20, 'mean_test_score')
y_pos = range(len(top20))
axes[1].barh(y_pos, top20['mean_test_score'].values,
             xerr=top20['std_test_score'].values,
             color='#534AB7', alpha=0.8, edgecolor='white',
             capsize=3)
labels_20 = [f"d={r['param_max_depth']} sp={r['param_min_samples_split']} "
              f"lf={r['param_min_samples_leaf']}"
              for _, r in top20.iterrows()]
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(labels_20, fontsize=7)
axes[1].set_xlabel('CV F1-Score (weighted) ± std')
axes[1].set_title('Top 20 Hyperparameter Combinations')

plt.suptitle('Grid Search Results — Decision Tree Hyperparameter Tuning', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3 Evaluate the best Decision Tree on the test set ────────────────────────
# WHY: grid_dt.best_estimator_ is the model that was automatically retrained
# on the full training set with the best hyperparameters. We evaluate it
# on the held-out test set — data it has NEVER seen.

dt_best   = grid_dt.best_estimator_
y_pred_dt = dt_best.predict(X_test)

acc_dt  = accuracy_score(y_test, y_pred_dt)
f1_dt   = f1_score(y_test, y_pred_dt, average='weighted')
pre_dt  = precision_score(y_test, y_pred_dt, average='weighted')
rec_dt  = recall_score(y_test, y_pred_dt, average='weighted')

print('=== Decision Tree — Best Model (after Grid Search) ===')
print(f'  Best params    : {grid_dt.best_params_}')
print(f'  Accuracy       : {acc_dt:.4f}  ({acc_dt*100:.2f}%)')
print(f'  Precision (w)  : {pre_dt:.4f}')
print(f'  Recall    (w)  : {rec_dt:.4f}')
print(f'  F1-Score  (w)  : {f1_dt:.4f}')
print(f'  Tree depth     : {dt_best.get_depth()}')
print(f'  Num leaves     : {dt_best.get_n_leaves()}')
print()
print('Per-class report:')
print(classification_report(y_test, y_pred_dt, target_names=LABELS))

In [ ]:
# ── 6.4 Visualise Decision Tree (top 3 levels) ────────────────────────────────
# WHY: Plotting the tree directly shows the if/else logic the model learned.
# Even with a complex deep tree, the top 3 levels reveal the most important
# splits — the features and thresholds that drive the majority of decisions.

fig, ax = plt.subplots(figsize=(22, 7))
plot_tree(
    dt_best, ax=ax,
    feature_names=FEATURE_COLS,
    class_names=LABELS,
    filled=True, rounded=True,
    max_depth=3, fontsize=7,
    impurity=False
)
ax.set_title(f'Decision Tree — Top 3 Levels  (full depth={dt_best.get_depth()})\n'
             f'Best params: {grid_dt.best_params_}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.5 Feature importance ────────────────────────────────────────────────────
# WHY: Decision Tree feature importances tell us which circuits and temporal
# features drove the classification decisions (measured by Gini impurity
# reduction across all splits). High importance = strong predictor.
# This directly tells a facility manager which appliances to monitor.

feat_imp = pd.Series(dt_best.feature_importances_, index=FEATURE_COLS).sort_values()
top15    = feat_imp.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors_fi = ['#BA7517' if 'hour' in f or 'day' in f or 'month' in f or 'weekend' in f
              else '#534AB7' for f in top15.index]
top15.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Feature Importance (Gini reduction)')
ax.set_title('Top 15 Features — Decision Tree (Grid Search best model)\n'
             'Purple = circuit features | Orange = temporal features')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.6 Confusion matrix ─────────────────────────────────────────────────────
# WHY: Accuracy alone hides which class pairs are confused.
# The confusion matrix shows whether the model makes 'adjacent' errors
# (Medium predicted as Low — minor mistake) or 'severe' errors
# (Low predicted as High — completely wrong tier).

cm_dt = confusion_matrix(y_test, y_pred_dt, labels=LABELS)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS,
            ax=ax, linewidths=0.5, annot_kws={'size': 11})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title(f'Confusion Matrix — Decision Tree\n'
             f'(Accuracy={acc_dt:.3f}  F1={f1_dt:.3f})')
plt.tight_layout()
plt.show()

---
## 7. Algorithm 3 — KNN + 5-Fold Cross-Validation
> **Why KNN:** K-Nearest Neighbours classifies a new data point by majority vote among its K nearest neighbours in the feature space. It makes no assumptions about the data distribution (non-parametric) and naturally captures local patterns — if a 1-minute snapshot looks like 5 previous High-tier snapshots, it predicts High.

> **Why Cross-Validation for tuning:** KNN has one key hyperparameter: K (number of neighbours). Too small (K=1) = overfitting to noise. Too large (K=100) = underfitting. 5-Fold Stratified CV reliably finds the K that generalises best without touching the test set.

In [ ]:
# ── 7.1 Cross-validation to find optimal K ────────────────────────────────────
# WHY: We test K=1 to 30 using 5-Fold Stratified CV on the training set.
# 'Stratified' ensures each fold preserves the class proportions —
# critical for fair CV with an ordered time-series target like energy_tier.
# weights='distance' means closer neighbours get more influence than farther ones.

K_VALUES = range(1, 31)
cv_means = []
cv_stds  = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Running 5-Fold CV for K = 1 to 30 ...')
for k in K_VALUES:
    knn_tmp = KNeighborsClassifier(n_neighbors=k, metric='euclidean',
                                    weights='distance')
    scores  = cross_val_score(knn_tmp, X_train, y_train,
                               cv=skf, scoring='f1_weighted')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f'  K={k:2d} | CV F1 = {scores.mean():.4f} ± {scores.std():.4f}')

best_k = K_VALUES[np.argmax(cv_means)]
print(f'\nBest K = {best_k}  (CV F1 = {max(cv_means):.4f})')

In [ ]:
# ── 7.2 Plot K vs CV F1 Score ─────────────────────────────────────────────────
# WHY: The shaded band (±1 std) shows how stable each K value is across folds.
# A wide band = high variance = the model is sensitive to which data is in the
# fold (unstable). A narrow band at the peak = the best K is robust.

cv_means_arr = np.array(cv_means)
cv_stds_arr  = np.array(cv_stds)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(K_VALUES, cv_means_arr, marker='o', color='#BA7517',
        linewidth=2, markersize=5)
ax.fill_between(K_VALUES,
                cv_means_arr - cv_stds_arr,
                cv_means_arr + cv_stds_arr,
                alpha=0.15, color='#BA7517', label='±1 std (stability band)')
ax.axvline(x=best_k, linestyle='--', color='#993C1D', alpha=0.8,
           label=f'Best K={best_k}  (F1={max(cv_means):.4f})')
ax.set_xlabel('K (Number of Neighbours)')
ax.set_ylabel('CV F1-Score (weighted)')
ax.set_title('KNN — K Selection via 5-Fold Cross-Validation\nShaded band = ±1 std across 5 folds')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Train final KNN with best K ───────────────────────────────────────────
# WHY weights='distance': instead of giving all K neighbours equal votes,
# closer neighbours get more weight (1/distance). This reduces the impact of
# neighbours that are far away and barely similar to the query point.
# metric='euclidean': standard Euclidean distance in the scaled feature space.

knn = KNeighborsClassifier(n_neighbors=best_k,
                            metric='euclidean',
                            weights='distance')
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn  = f1_score(y_test, y_pred_knn, average='weighted')
pre_knn = precision_score(y_test, y_pred_knn, average='weighted')
rec_knn = recall_score(y_test, y_pred_knn, average='weighted')

print(f'=== KNN (K={best_k}, weights=distance, metric=euclidean) ===')
print(f'  Accuracy       : {acc_knn:.4f}  ({acc_knn*100:.2f}%)')
print(f'  Precision (w)  : {pre_knn:.4f}')
print(f'  Recall    (w)  : {rec_knn:.4f}')
print(f'  F1-Score  (w)  : {f1_knn:.4f}')
print()
print('Per-class report:')
print(classification_report(y_test, y_pred_knn, target_names=LABELS))

In [ ]:
# ── 7.4 Confusion matrix ─────────────────────────────────────────────────────
# WHY: Comparing KNN's confusion matrix against Decision Tree's shows
# whether the two algorithms make the SAME mistakes or different ones.
# If they fail on different cases, an ensemble could combine them.

cm_knn = confusion_matrix(y_test, y_pred_knn, labels=LABELS)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges',
            xticklabels=LABELS, yticklabels=LABELS,
            ax=ax, linewidths=0.5, annot_kws={'size': 11})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title(f'Confusion Matrix — KNN (K={best_k})\n'
             f'(Accuracy={acc_knn:.3f}  F1={f1_knn:.3f})')
plt.tight_layout()
plt.show()

---
## 8. Final Comparison
> **Why compare here:** Each algorithm was tuned independently. Now we bring all results together on the same test set to answer the core research question: which algorithm best classifies energy tiers in a smart home, and what does each algorithm contribute uniquely?

In [ ]:
# ── 8.1 Summary metrics table ─────────────────────────────────────────────────
# WHY: A single table with all metrics side-by-side is the standard way to
# present algorithm comparisons in academic projects and industry reports.

summary = pd.DataFrame({
    'Algorithm'         : ['K-Means', 'Decision Tree (Grid Search)', f'KNN (K={best_k}, CV)'],
    'Type'              : ['Unsupervised', 'Supervised', 'Supervised'],
    'Tuning Method'     : ['Elbow + Silhouette', 'GridSearchCV (5-Fold)', '5-Fold Stratified CV'],
    'Primary Metric'    : [f'Purity={overall_purity:.4f}', f'F1={f1_dt:.4f}', f'F1={f1_knn:.4f}'],
    'Accuracy'          : ['N/A', f'{acc_dt:.4f}', f'{acc_knn:.4f}'],
    'Precision (w)'     : ['N/A', f'{pre_dt:.4f}', f'{pre_knn:.4f}'],
    'Recall (w)'        : ['N/A', f'{rec_dt:.4f}', f'{rec_knn:.4f}'],
    'Silhouette'        : [f'{sil_final:.4f}', 'N/A', 'N/A'],
    'Interpretable'     : ['Moderate', 'High ✓', 'Low'],
    'Needs Labels'      : ['No', 'Yes', 'Yes'],
    'Scalable'          : ['Medium', 'High ✓', 'Low (lazy learner)'],
})

print('='*110)
print('FINAL ALGORITHM COMPARISON — Smart* Home A Energy Classification')
print('='*110)
print(summary.to_string(index=False))

In [ ]:
# ── 8.2 Side-by-side bar chart — DT vs KNN ───────────────────────────────────
# WHY: Bar charts make the metric differences instantly visible.
# Error bars are not shown here since we are reporting test set metrics,
# not CV distributions — the comparison is deterministic at this point.

metrics    = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
dt_scores  = [acc_dt,  f1_dt,  pre_dt,  rec_dt]
knn_scores = [acc_knn, f1_knn, pre_knn, rec_knn]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, dt_scores,  width,
            label=f'Decision Tree (Grid Search)', color='#534AB7', alpha=0.88)
b2 = ax.bar(x + width/2, knn_scores, width,
            label=f'KNN (K={best_k})', color='#BA7517', alpha=0.88)
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Decision Tree (Grid Search) vs KNN — All Classification Metrics')
ax.axhline(y=0.9, linestyle='--', color='gray', alpha=0.35, linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.3 Side-by-side confusion matrices ──────────────────────────────────────
# WHY: Placing both matrices next to each other makes it easy to spot
# which tier each algorithm struggles with, and whether they struggle
# in the same places or in different places.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 11})
axes[0].set_title(f'Decision Tree (Grid Search)\nF1={f1_dt:.4f}  Acc={acc_dt:.4f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges',
            xticklabels=LABELS, yticklabels=LABELS,
            ax=axes[1], linewidths=0.5, annot_kws={'size': 11})
axes[1].set_title(f'KNN (K={best_k})\nF1={f1_knn:.4f}  Acc={acc_knn:.4f}')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.suptitle('Confusion Matrices Comparison — Where does each model make mistakes?', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4 Full comparison dashboard ────────────────────────────────────────────
# WHY: A single-figure dashboard brings together the key visuals from all
# three algorithms. This is the figure to include in the project report.

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.35)

# Panel 1 — K-Means PCA
ax1 = fig.add_subplot(gs[0, 0])
for c, color in enumerate(CLR3):
    mask = km_labels == c
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color,
                s=0.5, alpha=0.2, label=cluster_name_map[c])
ax1.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            c='black', s=100, marker='X', zorder=5)
ax1.set_title(f'K-Means (K=3)\nSilhouette={sil_final:.4f}  Purity={overall_purity:.4f}', fontsize=9)
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
ax1.legend(fontsize=6, markerscale=5)

# Panel 2 — DT Confusion Matrix
ax2 = fig.add_subplot(gs[0, 1])
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['L','M','H'], yticklabels=['L','M','H'],
            linewidths=0.5, cbar=False, annot_kws={'size': 10})
ax2.set_title(f'Decision Tree (Grid Search)\nF1={f1_dt:.4f}  Acc={acc_dt:.4f}', fontsize=9)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

# Panel 3 — KNN Confusion Matrix
ax3 = fig.add_subplot(gs[0, 2])
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges', ax=ax3,
            xticklabels=['L','M','H'], yticklabels=['L','M','H'],
            linewidths=0.5, cbar=False, annot_kws={'size': 10})
ax3.set_title(f'KNN (K={best_k}  5-Fold CV)\nF1={f1_knn:.4f}  Acc={acc_knn:.4f}', fontsize=9)
ax3.set_xlabel('Predicted'); ax3.set_ylabel('Actual')

# Panel 4 — Feature Importance (DT)
ax4 = fig.add_subplot(gs[1, 0])
top10 = feat_imp.nlargest(10).sort_values()
top10.plot(kind='barh', ax=ax4, color='#534AB7', edgecolor='white', linewidth=0.4)
ax4.set_title('Top 10 Features (Decision Tree)', fontsize=9)
ax4.set_xlabel('Importance')
ax4.tick_params(labelsize=7)

# Panel 5 — Hourly load by tier
ax5 = fig.add_subplot(gs[1, 1])
for tier, color in PALETTE.items():
    h = df_clean[df_clean['energy_tier']==tier].groupby('hour')['total_load_watts'].mean()
    ax5.plot(h.index, h.values, label=tier, color=color, linewidth=2)
ax5.set_xlabel('Hour'); ax5.set_ylabel('Avg Load (W)')
ax5.set_title('Load by Hour per Tier', fontsize=9)
ax5.legend(fontsize=7)

# Panel 6 — Metrics bar chart
ax6 = fig.add_subplot(gs[1, 2])
xp = np.arange(len(metrics)); w = 0.35
bb1 = ax6.bar(xp-w/2, dt_scores,  w, color='#534AB7', alpha=0.88, label='Decision Tree')
bb2 = ax6.bar(xp+w/2, knn_scores, w, color='#BA7517', alpha=0.88, label=f'KNN K={best_k}')
ax6.bar_label(bb1, fmt='%.2f', padding=2, fontsize=7)
ax6.bar_label(bb2, fmt='%.2f', padding=2, fontsize=7)
ax6.set_xticks(xp); ax6.set_xticklabels(metrics, fontsize=8)
ax6.set_ylim(0, 1.15); ax6.set_ylabel('Score')
ax6.set_title('Metric Comparison', fontsize=9)
ax6.legend(fontsize=7)

fig.suptitle('Smart* Home A — Full Algorithm Comparison Dashboard', fontsize=14, y=1.01)
plt.savefig('smartstar_comparison_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved as smartstar_comparison_dashboard.png')

---
## 9. Conclusions

### Preprocessing decisions summary
| Step | Technique | Why |
|---|---|---|
| Outlier handling | 99th percentile capping (Winsorization) | Preserves all records, reduces centroid distortion |
| Feature removal | Drop circuits with std < 5W | Near-constant features add noise, no signal |
| Scaling | StandardScaler (z-score) | Equalises circuit Watt ranges for K-Means and KNN |
| Dimensionality | PCA 2D (visualisation only) | Inspect cluster separation without affecting models |
| Split | Stratified 80/20 train/test | Preserves class proportions, prevents leakage |

### Tuning decisions summary
| Algorithm | Tuning method | What was tuned |
|---|---|---|
| K-Means | Elbow Method + Silhouette Score | Number of clusters K (tested 2–10) |
| Decision Tree | GridSearchCV (5-Fold, F1-weighted) | max_depth, min_samples_split, min_samples_leaf, class_weight |
| KNN | 5-Fold Stratified CV (F1-weighted) | Number of neighbours K (tested 1–30) |

### Algorithm comparison summary
| Dimension | K-Means | Decision Tree | KNN |
|---|---|---|---|
| Type | Unsupervised | Supervised | Supervised |
| Key metric | Silhouette + Cluster Purity | Accuracy + F1 | Accuracy + F1 |
| Needs labels | No ✓ | Yes | Yes |
| Interpretable | Moderate | High ✓ (explicit rules) | Low |
| Scalable | Medium | High ✓ | Low (lazy learner) |
| Best use case | Discover usage patterns | Automation trigger rules | Real-time state lookup |

### Key findings
1. **K-Means** validated that the Low/Medium/High energy tiers represent real natural structure in the data — the algorithm discovered them without ever seeing the labels.
2. **Decision Tree (Grid Search)** produced interpretable if/else rules directly actionable for smart building automation. Grid Search over 4 hyperparameters found a better model than manual depth tuning alone.
3. **KNN** is competitive in accuracy but computationally expensive at inference time — for every new 1-minute reading it must scan the full 105,984-record training set. Impractical for real-time IoT deployment.
4. The most important features across both classifiers are the **HVAC circuits (FurnaceHRV, DuctHeaterHRV)** and **temporal features (hour, dayofweek)** — confirming that heating system activity and time-of-day drive energy tier transitions in this home.